# PathSnap — Dataset Exploration

This notebook reproduces the first EDA stage using the supplied `train` and `student_test` partitions. It inspects the four files per trip, checks coordinate and timestamp quality, validates graph/route identifiers and generates an interactive Folium map. Evaluator-testing data is not part of this notebook.

The source schema comes from the assignment Dataset Exploration guide: `.track` = longitude/latitude/timestamp; `.nodes` = longitude/latitude; `.arcs` = from/to node IDs; `.route` = ordered arc IDs.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import IFrame, display

candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p.resolve() for p in candidates if (p / 'data' / 'raw').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Could not locate data/raw. Open the PathSnap repository in VS Code and copy train/ and student_test/ into data/raw/.')
DATA_ROOT = ROOT / 'data' / 'raw'
print('Project root:', ROOT)
print('Dataset root:', DATA_ROOT)
print('Training split exists:', (DATA_ROOT / 'train').is_dir())
print('Student-test split exists:', (DATA_ROOT / 'student_test').is_dir())

## 1. Inspect one trip

The row index is an identifier in the road network: node row index is `node_id`, arc row index is `arc_id`. A GPS observation row does not directly correspond to a node/arc row.

In [ ]:
train_dir = DATA_ROOT / 'train'
trip_dirs = sorted(p for p in train_dir.iterdir() if p.is_dir())
trip = trip_dirs[0]
trip_id = trip.name
track = pd.read_csv(trip / f'{trip_id}.track', sep=r'\\s+', header=None, names=['longitude', 'latitude', 'timestamp'])
nodes = pd.read_csv(trip / f'{trip_id}.nodes', sep=r'\\s+', header=None, names=['longitude', 'latitude'])
arcs = pd.read_csv(trip / f'{trip_id}.arcs', sep=r'\\s+', header=None, names=['from_node_id', 'to_node_id'])
route = pd.read_csv(trip / f'{trip_id}.route', sep=r'\\s+', header=None, names=['arc_id'])
print('Trip:', trip_id)
print('Rows — track:', len(track), '| nodes:', len(nodes), '| arcs:', len(arcs), '| route:', len(route))
display(track.head())
display(nodes.head())
display(arcs.head())
display(route.head())

## 2. Run the full dataset audit

The script writes a per-trip inventory and an aggregate JSON summary. Graph connectivity is computed on the undirected version of each trip's directed arc graph to identify weakly connected components.

In [ ]:
command = [sys.executable, 'src/data_exploration.py', '--data', str(DATA_ROOT), '--output', 'results', '--map', 'visualizations/sample_trip_map.html']
completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError('Dataset audit failed. Check the error output above.')
summary = json.loads((ROOT / 'results' / 'dataset_summary.json').read_text(encoding='utf-8'))
summary

In [ ]:
inventory = pd.read_csv(ROOT / 'results' / 'trip_inventory.csv')
display(inventory.groupby('partition')[['track_rows', 'nodes', 'arcs', 'route_rows']].agg(['count', 'sum', 'median']))

fig, ax = plt.subplots(figsize=(8, 4))
inventory.boxplot(column='track_rows', by='partition', ax=ax)
ax.set_title('GPS observations per trip')
ax.set_xlabel('Dataset partition')
ax.set_ylabel('GPS observation count')
fig.suptitle('')
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(inventory['max_time_gap_s'].dropna(), bins=30)
ax.set_title('Largest time gap per trip')
ax.set_xlabel('Maximum timestamp gap (seconds)')
ax.set_ylabel('Number of trips')
plt.show()

## 3. Folium map — observed GPS vs reference route

The reference route is reconstructed from the route's `arc_id` sequence, looked up in `.arcs`, then converted to coordinates through the `.nodes` row indices. This map is a reference-route visualisation, not a model prediction.

In [ ]:
map_path = ROOT / 'visualizations' / 'sample_trip_map.html'
display(IFrame(src=str(map_path), width='100%', height=550))